# **Session 01**

In [2]:
import torch
from torch import nn

class ToyNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.intermediate = nn.Identity()

    def forward(self, a):
        m = 2 * a
        m = self.intermediate(m)
        y = 3 * m + 1
        return y

model = ToyNetwork()
model.eval()

# Shape: [batch size, number of input coordinates]
a = torch.tensor([[2.0]])

with torch.no_grad():
    clean_output = model(a)

print("Clean output:", clean_output.item())
assert clean_output.item() == 13.0

Clean output: 13.0


In [3]:
captured = {}

def capture_m(module, inputs, output):
    captured["m"] = output.detach().clone()
    # Returning None leaves the module's output unchanged.

handle = model.intermediate.register_forward_hook(capture_m)

try:
    with torch.no_grad():
        output_with_capture = model(a)
finally:
    handle.remove()

print("Captured m:", captured["m"].item())
print("Output with capture:", output_with_capture.item())

torch.testing.assert_close(output_with_capture, clean_output)
assert captured["m"].item() == 4.0

Captured m: 4.0
Output with capture: 13.0


In [4]:
def identity_patch(module, inputs, output):
    return captured["m"].clone()

handle = model.intermediate.register_forward_hook(identity_patch)

try:
    with torch.no_grad():
        identity_output = model(a)
finally:
    handle.remove()

print("Output after identity patch:", identity_output.item())

torch.testing.assert_close(identity_output, clean_output)
assert captured["m"].item() == 4.0

Output after identity patch: 13.0


In [9]:
def patch_m_to_five(module, inputs, output):
    return torch.full_like(output, 5.0)

handle = model.intermediate.register_forward_hook(patch_m_to_five)

try:
    with torch.no_grad():
        patched_output = model(a)
finally:
    handle.remove()

# Run again after removing the hook.
with torch.no_grad():
    restored_output = model(a)

effect = patched_output - clean_output

print("Patched output:", patched_output.item())
print("Intervention effect:", effect.item())
print("Restored clean output:", restored_output.item())
print("Saved clean m:", captured["m"].item())
print("Effect (patched_output - clean_output):", effect.item())

assert patched_output.item() == 16.0
assert effect.item() == 3.0
torch.testing.assert_close(restored_output, clean_output)
assert captured["m"].item() == 4.0

Patched output: 16.0
Intervention effect: 3.0
Restored clean output: 13.0
Saved clean m: 4.0
Effect (patched_output - clean_output): 3.0


## NOTES
  -  I verified that I can capture an intermediate activation without changing the output, replace it with the same value, then change it from 4 to 5. The output changes from 13 to 16 as expected. Removing the hook restores 13, and the saved activation remains 4.
  -  Session 1 checks whether the intervention is implemented correctly. Session 2 checks whether the evaluation detects a known incorrect dependency.


# **Session 02**

In [2]:
import torch

In [3]:
def original(x):
    return 2 * x[..., 1:2]

def exact_replacement(x):
    return 2 * x[..., 1:2]

def shortcut_replacement(x):
    return 2 * x[..., 0:1]


# Each row is one example: [u, m].
x_clean = torch.tensor([
    [1.0, 1.0],
    [2.0, 2.0],
    [3.0, 3.0],
])

with torch.no_grad():
    true_outputs = original(x_clean)
    exact_outputs = exact_replacement(x_clean)
    shortcut_outputs = shortcut_replacement(x_clean)

print("Original:\n", true_outputs)
print("Exact replacement:\n", exact_outputs)
print("Shortcut:\n", shortcut_outputs)

torch.testing.assert_close(exact_outputs, true_outputs)
torch.testing.assert_close(shortcut_outputs, true_outputs)

Original:
 tensor([[2.],
        [4.],
        [6.]])
Exact replacement:
 tensor([[2.],
        [4.],
        [6.]])
Shortcut:
 tensor([[2.],
        [4.],
        [6.]])


In [4]:
# Make an independent copy so the clean inputs remain unchanged.
x_changed = x_clean.clone()
# We'll increase m by +2 in every exemple and leave u unchanged for each row is : [u, m]
x_changed[..., 1:2] += 2.0

with torch.no_grad():
    delta_true = original(x_changed) - original(x_clean)
    delta_exact = exact_replacement(x_changed) - exact_replacement(x_clean)
    delta_shortcut = shortcut_replacement(x_changed) - shortcut_replacement(x_clean)

    error_exact = delta_exact - delta_true
    error_shortcut = delta_shortcut - delta_true

print("Changed inputs:\n", x_changed)
print("True changes:\n", delta_true)
print("Exact replacement changes:\n", delta_exact)
print("Shortcut changes:\n", delta_shortcut)
print("Shortcut response errors:\n", error_shortcut)

# Check against manually calculated answers.
torch.testing.assert_close(delta_true, torch.full_like(delta_true, 4.0))
torch.testing.assert_close(error_exact, torch.zeros_like(error_exact))
torch.testing.assert_close(delta_shortcut, torch.zeros_like(delta_shortcut))
torch.testing.assert_close(error_shortcut, torch.full_like(error_shortcut, -4.0))

# Confirm that the clean inputs were not overwritten.
torch.testing.assert_close(x_clean, torch.tensor([
    [1.0, 1.0],
    [2.0, 2.0],
    [3.0, 3.0],
]))

Changed inputs:
 tensor([[1., 3.],
        [2., 4.],
        [3., 5.]])
True changes:
 tensor([[4.],
        [4.],
        [4.]])
Exact replacement changes:
 tensor([[4.],
        [4.],
        [4.]])
Shortcut changes:
 tensor([[0.],
        [0.],
        [0.]])
Shortcut response errors:
 tensor([[-4.],
        [-4.],
        [-4.]])


In [5]:
x_changed_u = x_clean.clone()
# We'll increase u by +2 in every exemple and leave m unchanged for each row is : [u,
x_changed_u[..., 0:1] += 2.0

with torch.no_grad():
    delta_true_u = original(x_changed_u) - original(x_clean)
    delta_exact_u = exact_replacement(x_changed_u) - exact_replacement(x_clean)
    delta_shortcut_u = shortcut_replacement(x_changed_u) - shortcut_replacement(x_clean)

    error_exact_u = delta_exact_u - delta_true_u
    error_shortcut_u = delta_shortcut_u - delta_true_u

print("Changed inputs:\n", x_changed_u)
print("True changes:\n", delta_true_u)
print("Exact replacement changes:\n", delta_exact_u)
print("Shortcut changes:\n", delta_shortcut_u)
print("Shortcut response errors:\n", error_shortcut_u)

torch.testing.assert_close(delta_true_u, torch.zeros_like(delta_true_u))
torch.testing.assert_close(error_exact_u, torch.zeros_like(error_exact_u))
torch.testing.assert_close(
    delta_shortcut_u, torch.full_like(delta_shortcut_u, 4.0)
)
torch.testing.assert_close(
    error_shortcut_u, torch.full_like(error_shortcut_u, 4.0)
)

Changed inputs:
 tensor([[3., 1.],
        [4., 2.],
        [5., 3.]])
True changes:
 tensor([[0.],
        [0.],
        [0.]])
Exact replacement changes:
 tensor([[0.],
        [0.],
        [0.]])
Shortcut changes:
 tensor([[4.],
        [4.],
        [4.]])
Shortcut response errors:
 tensor([[4.],
        [4.],
        [4.]])


In [6]:
def evaluate_response(f, g, x, x_changed):
    """Inputs: [batch, input_dim]. Outputs: [batch, output_dim]."""
    assert x.shape == x_changed.shape

    with torch.no_grad():
        f_clean = f(x)
        f_changed = f(x_changed)
        g_clean = g(x)
        g_changed = g(x_changed)

        # Prevent accidental broadcasting between incompatible outputs.
        assert f_clean.ndim == 2
        assert (
            f_clean.shape == f_changed.shape
            == g_clean.shape == g_changed.shape
        )

        delta_true = f_changed - f_clean
        delta_pred = g_changed - g_clean
        response_error = delta_pred - delta_true

        # Sum over output coordinates, giving one score per example.
        squared_error_per_example = response_error.square().sum(dim=-1)

        # Average over examples, giving one summary score.
        mean_response_squared_error = squared_error_per_example.mean()

    return {
        "delta_true": delta_true,
        "delta_pred": delta_pred,
        "response_error": response_error,
        "squared_error_per_example": squared_error_per_example,
        "mean_response_squared_error": mean_response_squared_error,
    }

In [7]:
replacements = {
    "Exact": exact_replacement,
    "Shortcut": shortcut_replacement,
}

interventions = {
    "Change m": x_changed,
    "Change u": x_changed_u,
}

results = {}

for edit_name, edited_inputs in interventions.items():
    for model_name, replacement in replacements.items():
        result = evaluate_response(
            original, replacement, x_clean, edited_inputs
        )
        results[(edit_name, model_name)] = result

        score = result["mean_response_squared_error"].item()
        print(f"{edit_name} | {model_name}: {score:.1f}")

        expected_score = 0.0 if model_name == "Exact" else 16.0
        assert score == expected_score

Change m | Exact: 0.0
Change m | Shortcut: 16.0
Change u | Exact: 0.0
Change u | Shortcut: 16.0


In [8]:
#check identity edits and decreases
# Prepare three shared intervention cases.
x_identity = x_clean.clone()

x_decrease_m = x_clean.clone()
x_decrease_m[..., 1:2] -= 2.0

x_decrease_u = x_clean.clone()
x_decrease_u[..., 0:1] -= 2.0

# Each case contains:
# (edited inputs, expected true change, expected shortcut change)
test_cases = {
    "Identity":   (x_identity,    0.0,  0.0),
    "Decrease m": (x_decrease_m, -4.0,  0.0),
    "Decrease u": (x_decrease_u,  0.0, -4.0),
}

for case_name, (edited_inputs, true_change, shortcut_change) in test_cases.items():
    for model_name, replacement in replacements.items():
        result = evaluate_response(
            original, replacement, x_clean, edited_inputs
        )

        expected_prediction = (
            true_change if model_name == "Exact" else shortcut_change
        )
        expected_error = expected_prediction - true_change

        torch.testing.assert_close(
            result["delta_true"],
            torch.full_like(result["delta_true"], true_change),
        )
        torch.testing.assert_close(
            result["delta_pred"],
            torch.full_like(result["delta_pred"], expected_prediction),
        )
        torch.testing.assert_close(
            result["response_error"],
            torch.full_like(result["response_error"], expected_error),
        )

        score = result["mean_response_squared_error"].item()
        assert score == expected_error ** 2

        print(f"{case_name} | {model_name}: {score:.1f}")

print("All checks passed.")

Identity | Exact: 0.0
Identity | Shortcut: 0.0
Decrease m | Exact: 0.0
Decrease m | Shortcut: 16.0
Decrease u | Exact: 0.0
Decrease u | Shortcut: 16.0
All checks passed.


In [9]:
#test a nonlinear computation, ReLU(m).
def original_relu(x):
    return torch.relu(x[..., 1:2])

def exact_relu(x):
    return torch.relu(x[..., 1:2])

def shortcut_relu(x):
    return torch.relu(x[..., 0:1])


# Coordinates are still [u, m].
x_relu = torch.tensor([
    [-3.0, -3.0],
    [-1.0, -1.0],
    [ 1.0,  1.0],
])

x_relu_changed = x_relu.clone()
x_relu_changed[..., 1:2] += 2.0

# Independently calculated from the table above.
expected_true_change = torch.tensor([
    [0.0],
    [1.0],
    [2.0],
])

relu_results = {}

for name, replacement in {
    "Exact": exact_relu,
    "Shortcut": shortcut_relu,
}.items():
    # Both controls should match the original on these clean inputs.
    torch.testing.assert_close(
        replacement(x_relu), original_relu(x_relu)
    )

    result = evaluate_response(
        original_relu, replacement, x_relu, x_relu_changed
    )
    relu_results[name] = result

    torch.testing.assert_close(
        result["delta_true"], expected_true_change
    )

    expected_prediction = (
        expected_true_change
        if name == "Exact"
        else torch.zeros_like(expected_true_change)
    )

    torch.testing.assert_close(
        result["delta_pred"], expected_prediction
    )
    torch.testing.assert_close(
        result["response_error"],
        expected_prediction - expected_true_change,
    )

    print(
        f"{name} predicted changes:",
        result["delta_pred"].flatten().tolist(),
    )
    print(
        f"{name} mean squared response error:",
        round(result["mean_response_squared_error"].item(), 4),
    )

Exact predicted changes: [0.0, 1.0, 2.0]
Exact mean squared response error: 0.0
Shortcut predicted changes: [0.0, 0.0, 0.0]
Shortcut mean squared response error: 1.6667


In [10]:
#test redundant pathways—remove either one, then both.
def original_or(x):
    s = x[..., 0:1] + x[..., 1:2]
    return s - torch.relu(s - 1)

def exact_or(x):
    # Independent implementation, equivalent on binary inputs.
    return torch.maximum(x[..., 0:1], x[..., 1:2])

def incorrect_constant(x):
    return torch.ones_like(x[..., 0:1])


# Verify the original and exact control on all four binary states.
binary_states = torch.tensor([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])
expected_outputs = torch.tensor([[0.0], [1.0], [1.0], [1.0]])

torch.testing.assert_close(original_or(binary_states), expected_outputs)
torch.testing.assert_close(exact_or(binary_states), expected_outputs)

# Three experiments, each starting with both pathways active.
x_both = torch.tensor([
    [1.0, 1.0],
    [1.0, 1.0],
    [1.0, 1.0],
])
x_removed = torch.tensor([
    [0.0, 1.0],  # Remove m1 only.
    [1.0, 0.0],  # Remove m2 only.
    [0.0, 0.0],  # Remove both.
])

expected_changes = torch.tensor([[0.0], [0.0], [-1.0]])

for name, replacement in {
    "Exact": exact_or,
    "Constant": incorrect_constant,
}.items():
    result = evaluate_response(
        original_or, replacement, x_both, x_removed
    )

    expected_prediction = (
        expected_changes
        if name == "Exact"
        else torch.zeros_like(expected_changes)
    )

    torch.testing.assert_close(result["delta_true"], expected_changes)
    torch.testing.assert_close(result["delta_pred"], expected_prediction)
    torch.testing.assert_close(
        result["squared_error_per_example"],
        (expected_prediction - expected_changes).square().sum(dim=-1),
    )

    print(f"{name} predicted changes:",
          result["delta_pred"].flatten().tolist())
    print(f"{name} squared errors:",
          result["squared_error_per_example"].tolist())

Exact predicted changes: [0.0, 0.0, -1.0]
Exact squared errors: [0.0, 0.0, 0.0]
Constant predicted changes: [0.0, 0.0, 0.0]
Constant squared errors: [0.0, 0.0, 1.0]


Let’s close session 2 by reviewing what your toy checks establish. If all the assertions passed, your evaluator produced the expected results in these cases:

| Check | Expected result | What it checks |
|---|---|---|
| Identity edit | Zero response | Unchanged inputs produce no predicted change |
| Exact replacement | Zero response error | Correct responses receive the correct score |
| Change `m` in the linear example | Shortcut misses the effect | Detection of a missing dependency |
| Change `u` only | Shortcut invents an effect | Detection of an incorrect dependency |
| Decrease a coordinate | Correct signed responses | The evaluator handles both intervention directions |
| ReLU examples | True changes of `0`, `1`, and `2` | Different nonlinear responses are measured correctly |
| Redundant pathways | Joint removal exposes the constant control | Individual tests can miss a failure |
